In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import xgboost as xgb

#Professional Configuration
DEVICE = torch.device("cpu")
SEED = 42
BATCH_SIZE = 64
EPOCHS = 40

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(" Environment Configured.         Device:", DEVICE)

 Environment Configured.         Device: cpu


In [2]:
#Load Data
df = pd.read_csv('../data/train_processed.csv')
visual_features = np.load('../data/image_features.npy')

#Feature Engineering
df['age'] = 2024 - df['yr_built']
df['lot_to_living_ratio'] = df['sqft_lot'] / df['sqft_living']

df['sqft_per_room'] = df['sqft_living'] / (df['bedrooms'] + df['bathrooms'] + 1)

#Define Columns

cat_col = 'cluster_id' 

cont_cols = [
    'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 
    'waterfront', 'view', 'condition', 'grade', 'sqft_above', 
    'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long',
    'sqft_living15', 'sqft_lot15', 'dist_to_downtown', 
    'age', 'lot_to_living_ratio', 'sqft_per_room'
]
target = 'log_price'

#Preprocessing
label_enc = LabelEncoder()
df[cat_col] = label_enc.fit_transform(df[cat_col])
num_clusters = len(label_enc.classes_)
scaler = StandardScaler()
X_cont = scaler.fit_transform(df[cont_cols])
X_cat = df[cat_col].values
y = df[target].values
indices = np.arange(len(df))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
X_cont_train, X_cont_val = X_cont[train_idx], X_cont[val_idx]
X_cat_train, X_cat_val = X_cat[train_idx], X_cat[val_idx]
vis_train, vis_val = visual_features[train_idx], visual_features[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f" Data Prepared. Training samples: {len(train_idx)}")
print(f" Visual Features Shape: {vis_train.shape}")

 Data Prepared. Training samples: 12960
 Visual Features Shape: (12960, 512)


In [3]:
class RealEstateDataset(Dataset):
    def __init__(self, cont, cat, vis, y):
        self.cont = torch.FloatTensor(cont)
        self.cat = torch.LongTensor(cat)
        self.vis = torch.FloatTensor(vis)
        self.y = torch.FloatTensor(y)
        
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (self.cont[i], self.cat[i], self.vis[i]), self.y[i]

class ResidualFusionNet(nn.Module):
    def __init__(self, num_cont, num_clusters, emb_dim=10, vis_dim=512):
        super().__init__()
        
        #Tabular Branch
        self.cluster_emb = nn.Embedding(num_clusters, emb_dim)
        
        self.tab_layer = nn.Sequential(
            nn.Linear(num_cont + emb_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # Visual Branch
        self.vis_layer = nn.Sequential(
            nn.Linear(vis_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3)
        )
        
        # Fusion Layer
        self.fusion = nn.Sequential(
            nn.Linear(128 + 128, 128),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        self.head = nn.Linear(128 + 128, 1) 

    def forward(self, cont, cat, vis):
        emb = self.cluster_emb(cat)
        
        # Tabular Process
        tab_in = torch.cat([cont, emb], dim=1)
        tab_feat = self.tab_layer(tab_in)
        
        # Visual Process
        vis_feat = self.vis_layer(vis)
        
        # Combine
        combined = torch.cat([tab_feat, vis_feat], dim=1)
        fused = self.fusion(combined)
        
        # Skip Connection: Fuse + Original Tabular Features
        final_in = torch.cat([fused, tab_feat], dim=1)
        
        return self.head(final_in)

print(" Residual Fusion Architecture Initialized.")

 Residual Fusion Architecture Initialized.


In [4]:
#Loaders
train_ds = RealEstateDataset(X_cont_train, X_cat_train, vis_train, y_train)
val_ds = RealEstateDataset(X_cont_val, X_cat_val, vis_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False)

# Model & Optimization
model = ResidualFusionNet(num_cont=X_cont.shape[1], num_clusters=num_clusters).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = nn.HuberLoss(delta=1.0) 
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=0.01, steps_per_epoch=len(train_loader), epochs=EPOCHS
)

#Training Loop
best_loss = float('inf')
print(f" Starting Training for {EPOCHS} Epochs...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    
    for (c, cat, v), y_batch in train_loader:
        c, cat, v, y_batch = c.to(DEVICE), cat.to(DEVICE), v.to(DEVICE), y_batch.to(DEVICE)
        
        optimizer.zero_grad()
        out = model(c, cat, v).squeeze()
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for (c, cat, v), y_batch in val_loader:
            c, cat, v, y_batch = c.to(DEVICE), cat.to(DEVICE), v.to(DEVICE), y_batch.to(DEVICE)
            out = model(c, cat, v).squeeze()
            val_loss += criterion(out, y_batch).item()
            
    val_loss /= len(val_loader)
    
    # Save 
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_fusion_model.pth')
        
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | Val Loss: {val_loss:.5f} | Best: {best_loss:.5f}")

print(" Training Completed.")

 Starting Training for 40 Epochs...
Epoch 05 | Val Loss: 0.05237 | Best: 0.05237
Epoch 10 | Val Loss: 0.03150 | Best: 0.03150
Epoch 15 | Val Loss: 0.02223 | Best: 0.02223
Epoch 20 | Val Loss: 0.02541 | Best: 0.02223
Epoch 25 | Val Loss: 0.02001 | Best: 0.01898
Epoch 30 | Val Loss: 0.01890 | Best: 0.01890
Epoch 35 | Val Loss: 0.02203 | Best: 0.01828
Epoch 40 | Val Loss: 0.01861 | Best: 0.01798
 Training Completed.


In [5]:
print(" Training XGBoost ...")

#Prepare tabular data for XGBoost (Cont + Cat)
X_train_xgb = np.hstack([X_cont_train, X_cat_train.reshape(-1, 1)])
X_val_xgb = np.hstack([X_cont_val, X_cat_val.reshape(-1, 1)])

#Params
xgb_model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.7,
    colsample_bytree=0.7,
    n_jobs=-1,
    random_state=SEED,
    early_stopping_rounds=50 
)

#Train
xgb_model.fit(
    X_train_xgb, y_train,
    eval_set=[(X_val_xgb, y_val)],
    verbose=False
)

#Generate Predictions & Score
xgb_preds = xgb_model.predict(X_val_xgb)
xgb_r2 = r2_score(np.expm1(y_val), np.expm1(xgb_preds))
print(f" XGBoost Solo R²: {xgb_r2:.5f}")

 Training XGBoost ...
 XGBoost Solo R²: 0.89092


In [6]:
#Load NN
model.load_state_dict(torch.load('best_fusion_model.pth'))
model.eval()

#Get NN Predictions
nn_preds = []
with torch.no_grad():
    for (c, cat, v), _ in val_loader:
        c, cat, v = c.to(DEVICE), cat.to(DEVICE), v.to(DEVICE)
        out = model(c, cat, v).squeeze()
        nn_preds.extend(out.cpu().numpy())
nn_preds = np.array(nn_preds)

#Optimize the Blend Weight
best_score = 0
best_w = 0

y_true = np.expm1(y_val)


for w in np.linspace(0, 1, 101):
    # Weighted Average
    blend_log = (w * xgb_preds) + ((1-w) * nn_preds)
    blend_price = np.expm1(blend_log)
    
    score = r2_score(y_true, blend_price)
    
    if score > best_score:
        best_score = score
        best_w = w

#Final Report
final_mae = mean_absolute_error(y_true, np.expm1((best_w * xgb_preds) + ((1-best_w) * nn_preds)))


print(f"       FINAL  RESULTS        ")

print(f"Optimal Mix     : {best_w*100:.0f}% XGBoost + {(1-best_w)*100:.0f}% NeuralNet")
print(f"NeuralNet R²    : {r2_score(y_true, np.expm1(nn_preds)):.5f}")
print(f"XGBoost R²      : {xgb_r2:.5f}")
print(f"------------------------------------------")
print(f"FINAL ENSEMBLE R²: {best_score:.5f}")
print(f"FINAL MAE        : ${final_mae:,.2f}")


       FINAL  RESULTS        
Optimal Mix     : 85% XGBoost + 15% NeuralNet
NeuralNet R²    : 0.84571
XGBoost R²      : 0.89092
------------------------------------------
FINAL ENSEMBLE R²: 0.89219
FINAL MAE        : $61,744.70
